# M04A · Biological Plots & Their Applications
**Programming for Computational Biology (Python)** · Author: Prateek Paul · Last updated: 2026-10-29

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meprateekpaul/ProgrammingCB/blob/main/modules/M04_Visualisation/M04A_Biological_Plots_Theory_Solutions.ipynb)

⏱ ~45 min · Difficulty: ★☆☆ · Prerequisites: M01A (Core Python), M01B (Sequence Manipulation)


## 1. Learning Outcomes

By the end of this notebook, you will be able to:

- **Classify**: Identify and explain the primary visualization modalities used across genomics, proteomics, phylogenetics, population genetics, and structural biology.
- **Select**: Navigate the chart selection decision matrix to select mathematically and perceptually appropriate plots for any experimental data structure.
- **Diagnose**: Detect deceptive and flawed visual designs in biological literature (3D projections, truncated axes, rainbow colormaps, dynamite plunger plots).
- **Formulate**: Apply principles of accessible scientific communication including colorblind-safe palettes (viridis, plasma, cividis) and high-contrast styling.

## 2. Prerequisites


Before starting this notebook, you should be comfortable with:
- Basic Python syntax, variables, lists, dictionaries, and functions ([`M01A`](file:///modules/M01_Python_Basics/M01A_Core_Python.ipynb)).
- Reading and manipulating biological sequence identifiers and summary counts.
- Fundamental statistics concepts: mean, standard deviation, median, interquartile range (IQR).


In [ ]:
# --- Setup -------------------------------------------------------
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install biopython seaborn requests beautifulsoup4 lxml python-dotenv
    !git clone -q https://github.com/meprateekpaul/ProgrammingCB.git
    ROOT = Path("ProgrammingCB")
else:
    # Resolve repository root robustly from modules/ or repo root
    p = Path.cwd()
    while p != p.parent and not (p / "COURSE_PLAN.md").exists():
        p = p.parent
    ROOT = p if (p / "COURSE_PLAN.md").exists() else Path.cwd().resolve()

DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs" / "M04A"
OUT_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")
print(f"DATA_DIR: {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"OUT_DIR:  {OUT_DIR} (ready)")

## 4A.1 Biological Plot Catalogue by Sub-discipline

### 4.1.1 Theory & Biological Context

Computational biology encompasses vastly distinct data modalities—from 1D nucleotide sequences and 2D expression matrices to 3D macromolecular atomic coordinates and phylogenetic trees. Choosing the appropriate visual representation is critical: a poorly chosen plot obscures true biological mechanisms, while an effective plot immediately communicates discovery.

Here is the foundational catalogue of biological plots classified across the major sub-disciplines:

### 1. Genomics and Transcriptomics
- **Ideograms / Karyotypes**: Chromosome-wide structural overviews showing cytogenetic bands, copy number alterations (CNAs), and chromosomal rearrangements.
- **Circos Plots**: Circular multi-track visualisations showing structural genomic rearrangements, synteny across species, and multi-omic overlays.
- **Manhattan Plots**: Used in Genome-Wide Association Studies (GWAS). Genomic coordinate along the x-axis, $-\log_{10}(p\text{-value})$ on the y-axis, highlighting trait-associated single nucleotide polymorphisms (SNPs) rising above genome-wide significance ($p < 5 \times 10^{-8}$).
- **Volcano Plots**: Differential expression standard. Plots magnitude of change ($\log_2 \text{Fold Change}$) on the x-axis against statistical significance ($-\log_{10} p\text{-value}$) on the y-axis.
- **MA Plots**: Ratio-intensity plot. Plots average expression ($M = \log_2(R/G)$ vs $A = \frac{1}{2}\log_2(RG)$) to detect intensity-dependent expression biases.
- **RNA-seq Sashimi Plots**: Visualizes splice junctions, exon inclusion/skipping levels, and read coverage depth across genomic coordinates.
- **Clustered Expression Heatmaps**: Visualizes gene expression matrices with hierarchical clustering trees (dendrograms) grouping co-expressed genes and similar biological conditions.

### 2. Proteomics and Metabolomics
- **MS/MS Spectra**: Intensity vs mass-to-charge ratio ($m/z$) with fragment ion annotations ($b$-ions, $y$-ions) for peptide sequencing.
- **Peptide Mass Fingerprints (PMF)**: Masses of proteolytic peptides matched against theoretical digestion databases.
- **Pathway Enrichment Bubble Charts**: Gene Ontology (GO) and KEGG pathways arranged by enrichment factor, bubble size proportional to gene count, colored by adjusted $p$-value.

### 3. Phylogenetics and Evolutionary Biology
- **Phylogenetic Trees**:
  - *Cladograms*: Show branching order (topology) without meaning in branch length.
  - *Phylograms*: Branch length is proportional to evolutionary distance (substitutions per site).
  - *Chronograms*: Branch length is calibrated to geological or chronological time.
- **Dot Plots**: 2D synteny and sequence alignment matrix revealing insertions, deletions, duplications, and inversions between two genomic sequences.
- **$dN/dS$ Ratio Plots**: Ratio of nonsynonymous to synonymous substitution rates along coding sequences to detect positive ($dN/dS > 1$), neutral ($= 1$), or purifying ($< 1$) selection.

### 4. Population Genetics and Ecology
- **Dimensionality Reduction (PCA, t-SNE, UMAP)**: Visualizes genetic variation across individuals or cell clusters in 2D/3D embeddings.
- **Admixture / Structure Bar Plots**: Stacked horizontal or vertical bars representing the fractional ancestry proportions of individuals across $K$ ancestral populations.
- **Rarefaction Curves**: Species richness plotted as a function of the number of individuals or reads sampled to assess sampling sufficiency.

### 5. Structural Biology
- **Ramachandran Plots**: Scatter plot of backbone dihedral torsion angles $\phi$ (phi) vs $\psi$ (psi) highlighting energetically favored secondary structure regions (alpha helices, beta sheets).
- **Residue Contact Maps**: 2D symmetric binary or distance matrix showing inter-residue Euclidean distances ($C_\alpha - C_\alpha$) within a 3D folded protein.
- **RMSF Plots**: Root Mean Square Fluctuation per residue over a molecular dynamics trajectory showing flexible loops vs rigid cores.

### 4.1.2 Worked Example

In [ ]:
# Biological metadata inspection and plot mapping
# Let's inspect data/tables/gene_annotation.csv to explore biological properties
annotation_path = DATA_DIR / "tables" / "gene_annotation.csv"
df_annot = pd.read_csv(annotation_path)

print(f"Loaded {len(df_annot)} annotated genes:")
print(df_annot.head(5))

# Compute gene lengths (end - start)
df_annot["length_kb"] = (df_annot["end"] - df_annot["start"]) / 1000.0

# Dictionary mapping biological questions to plot recommendations
BIO_PLOT_CATALOGUE = {
    "gene_length_distribution": {
        "plot": "Histogram with KDE",
        "rationale": "Shows distribution shape, skewness, and identifies multi-modal gene length populations.",
        "field": "Genomics"
    },
    "expression_comparison_across_tissues": {
        "plot": "Box plot or Violin plot",
        "rationale": "Displays median, spread, and outlier expression without assuming normality.",
        "field": "Transcriptomics"
    },
    "gwas_significant_snps": {
        "plot": "Manhattan plot",
        "rationale": "Maps -log10(p) across chromosomal positions with genome-wide threshold lines.",
        "field": "Genomics"
    },
    "protein_backbone_validity": {
        "plot": "Ramachandran plot",
        "rationale": "Plots phi vs psi dihedral angles against sterically allowed energy contours.",
        "field": "Structural Biology"
    }
}

for q, info in BIO_PLOT_CATALOGUE.items():
    print(f"-> Question: {q}")
    print(f"   Recommended: {info['plot']} [{info['field']}]")

### 4.1.3 ✏️ Try It: Biological Query Classifier

Write a function `classify_biological_query(query_keyword: str) -> str` that inspects a biological research query and returns the appropriate plot type:
- If query mentions `"gwas"` or `"variant association"` -> return `"Manhattan plot"`
- If query mentions `"differential expression"` or `"fold change"` -> return `"Volcano plot"`
- If query mentions `"torsion"` or `"phi psi"` or `"dihedral"` -> return `"Ramachandran plot"`
- If query mentions `"ancestry"` or `"admixture"` -> return `"Admixture bar plot"`
- If query mentions `"synteny"` or `"sequence match"` -> return `"Dot plot"`
- Otherwise return `"Unknown"`

In [ ]:
def classify_biological_query(query_keyword: str) -> str:
    q = query_keyword.lower()
    if "gwas" in q or "variant association" in q:
        return "Manhattan plot"
    elif "differential expression" in q or "fold change" in q:
        return "Volcano plot"
    elif "torsion" in q or "phi psi" in q or "dihedral" in q:
        return "Ramachandran plot"
    elif "ancestry" in q or "admixture" in q:
        return "Admixture bar plot"
    elif "synteny" in q or "sequence match" in q:
        return "Dot plot"
    return "Unknown"

#### 💡 Explanation & Design Notes

We normalize the search string to lowercase using `.lower()` and check for domain-specific terminology. In computational biology workflows, automated metadata classifiers frequently use keyword dictionaries to auto-suggest appropriate visualisation pipelines for incoming assay results.

In [ ]:
# ✅ Test — run me. No output means you passed.
assert classify_biological_query("Identify GWAS risk loci") == "Manhattan plot"
assert classify_biological_query("volcano plot of differential expression") == "Volcano plot"
assert classify_biological_query("Validate protein phi psi dihedral angles") == "Ramachandran plot"
assert classify_biological_query("Population admixture and ancestry fractions") == "Admixture bar plot"
assert classify_biological_query("Analyze genome synteny between human and chimp") == "Dot plot"
assert classify_biological_query("Unknown query") == "Unknown"
print("Section 4A.1 passed ✔")

### 4.1.4 ⚠️ Common Mistakes

- **Choosing by aesthetic rather than data distribution**: Do not choose a 3D surface plot simply because it looks sophisticated when a 2D heatmap or contour plot conveys the exact values without occlusion.
- **Confusing Cladogram with Phylogram**: In a cladogram, branch lengths are arbitrary and represent only branching relationships. In a phylogram, branch lengths encode genetic substitutions per site or time.
- **Using Pie Charts for High-Dimensional Categories**: Pie charts with more than 4-5 slices make angle comparison visually unreliable due to human cognitive limits in angle estimation.

## 4A.2 The Chart Selection Decision Matrix

### 4.2.1 Theory & Biological Context

When planning a visualization, begin with the **scientific question** and the **mathematical nature of the variables**, NOT the plotting software. 

### Fundamental Variable Types in Computational Biology:
1. **Nominal Categorical**: Discrete unranked labels (e.g., Gene symbols `TP53`, `EGFR`; Tissue types `Brain`, `Liver`; Chromosomes `chr1`, `chr2`).
2. **Ordinal Categorical**: Ranked categories (e.g., Clinical stages `Stage I`, `Stage II`, `Stage III`; Variant pathogenicity `Benign`, `VUS`, `Pathogenic`).
3. **Discrete Quantitative**: Integer counts (e.g., Read counts, Exon counts, Mutation burden).
4. **Continuous Quantitative**: Real-valued measurements (e.g., TPM expression, Binding affinity $K_d$, Molecular weight, $\log_2$ fold change).

### The Chart Selection Decision Matrix

| Question you are asking | Primary Data Types | Recommended Plot | Biology Example |
|---|---|---|---|
| **Compare values across discrete categories** | 1 Nominal + 1 Continuous | **Bar plot** | Gene count per chromosome |
| **Distribution shape of a single continuous variable** | 1 Continuous | **Histogram / Density (KDE)** | Read length distribution, GC% distribution |
| **Parts of a whole (2–4 categories only)** | 1 Categorical + Proportions | **Pie / Donut chart** | Nucleotide composition (A, C, G, T) |
| **Compare distributions with medians & outliers** | 1 Nominal + 1 Continuous | **Box plot** | Expression of gene across 5 tissues |
| **Compare multimodal distribution shapes across groups** | 1 Nominal + 1 Continuous | **Violin plot** | Single-cell marker gene expression per cluster |
| **Matrix of pairwise continuous values** | 2 Nominal + 1 Continuous (grid) | **Heatmap** | Gene $\times$ Sample expression matrix |
| **Relationship & correlation between two continuous variables** | 2 Continuous | **Scatter plot** | Gene expression vs promoter GC content |
| **Magnitude of change vs statistical significance** | 2 Continuous (Fold Change + p-val) | **Volcano plot** | RNA-seq differential expression |
| **Proportions across two nested categorical groups** | 2 Categorical + Frequencies | **Stacked / Grouped Bar** | Microbial phylum abundance across patient cohorts |

### 4.2.2 Worked Example

In [ ]:
# Implementing a programmatic chart recommendation engine
def recommend_chart_type(var_x_type: str, var_y_type: str, goal: str) -> str:
    """
    Recommends chart type based on variable types and scientific goal.
    """
    if goal == "correlation" and var_x_type == "continuous" and var_y_type == "continuous":
        return "Scatter plot"
    elif goal == "significance_vs_magnitude":
        return "Volcano plot"
    elif goal == "distribution" and var_y_type is None:
        return "Histogram or KDE"
    elif goal == "compare_distributions" and var_x_type == "categorical" and var_y_type == "continuous":
        return "Box plot or Violin plot"
    elif goal == "composition" and var_x_type == "categorical":
        return "Donut chart or Stacked Bar"
    elif goal == "matrix_values":
        return "Heatmap"
    elif goal == "compare_counts" and var_x_type == "categorical" and var_y_type == "numeric":
        return "Bar plot"
    return "Custom multi-panel figure"

# Test cases for biological data
print("TPM expression across 4 tissues:", recommend_chart_type("categorical", "continuous", "compare_distributions"))
print("Gene counts per chromosome:", recommend_chart_type("categorical", "numeric", "compare_counts"))
print("GC% vs RNA expression:", recommend_chart_type("continuous", "continuous", "correlation"))

### 4.2.3 ✏️ Try It: Chart Choice Validator

Write a function `validate_chart_choice(chart_type: str, num_categories: int, has_continuous_values: bool) -> tuple[bool, str]`:
Rules:
1. If `chart_type == "pie"` and `num_categories > 5`: return `(False, "Pie chart has too many slices; use a horizontal bar chart")`.
2. If `chart_type == "bar"` and not `has_continuous_values`: return `(False, "Bar plot requires numeric values; use contingency table or frequency count")`.
3. If `chart_type == "scatter"` and `num_categories > 0`: return `(False, "Scatter plot requires two continuous variables")`.
4. Otherwise return `(True, "Valid chart selection")`.

In [ ]:
def validate_chart_choice(chart_type: str, num_categories: int, has_continuous_values: bool) -> tuple[bool, str]:
    ct = chart_type.lower()
    if ct == "pie" and num_categories > 5:
        return (False, "Pie chart has too many slices; use a horizontal bar chart")
    if ct == "bar" and not has_continuous_values:
        return (False, "Bar plot requires numeric values; use contingency table or frequency count")
    if ct == "scatter" and num_categories > 0:
        return (False, "Scatter plot requires two continuous variables")
    return (True, "Valid chart selection")

#### 💡 Explanation & Design Notes

Validating chart parameters programmatically enforces good visualization hygiene before any pixels are drawn. Cognitive perception research demonstrates that human readers struggle to compare non-adjacent angles in pie charts with more than 5 slices.

In [ ]:
# ✅ Test — run me. No output means you passed.
ok, msg = validate_chart_choice("pie", num_categories=8, has_continuous_values=True)
assert not ok and "too many slices" in msg

ok, msg = validate_chart_choice("bar", num_categories=4, has_continuous_values=False)
assert not ok and "numeric values" in msg

ok, msg = validate_chart_choice("scatter", num_categories=3, has_continuous_values=True)
assert not ok and "continuous variables" in msg

ok, msg = validate_chart_choice("box", num_categories=4, has_continuous_values=True)
assert ok and "Valid" in msg
print("Section 4A.2 passed ✔")

### 4.2.4 ⚠️ Common Mistakes

- **Using line charts for nominal categories**: Connecting discrete categories (like Brain, Liver, Lung) with a line falsely implies continuous transition or time progression. Use separate bars or dots instead.
- **Truncating the y-axis in bar plots**: Bars encode magnitude by area and height. If the y-axis starts at 50 instead of 0, a 5% difference looks like a 500% difference, misleading the reader.
- **Using box plots for very small sample sizes (N < 5)**: Calculating quartiles and medians on fewer than 5 points is statistically meaningless; plot raw individual data points directly.

## 4A.3 Bad Plots in Biology & How to Fix Them

### 4.3.1 Theory & Biological Context

Scientific literature is unfortunately filled with deceptive, unreadable, or flawed figures. As a computational biologist, your goal is truthful, reproducible, and accessible data communication.

### The Four Major Visual Sins in Biology:

#### 1. 3D Charts (3D Pies, 3D Bars, 3D Ribbons)
- **The Flaw**: Perspective projection distorts geometric proportions. Slices angled toward the viewer appear artificially larger than identical or larger slices in the background.
- **The Fix**: Always use flat 2D representations.

#### 2. Rainbow Colormaps (`jet`, `rainbow`)
- **The Flaw**: The human eye perceives luminance non-linearly. In `jet`, there is an artificial yellow/cyan brightness spike where the data has a completely smooth gradient. Furthermore, 8% of male readers with red-green color vision deficiency (deuteranopia/protanopia) cannot distinguish red and green endpoints.
- **The Fix**: Perceptually uniform colormaps (`viridis`, `plasma`, `magma`, `cividis`). Their luminance increases monotonically with data value, ensuring accurate perception in color, greyscale print, and under all forms of colorblindness.

#### 3. Truncated / Broken y-Axes on Bar Charts
- **The Flaw**: A bar plot encodes value by physical length from the baseline. Setting $y_{\min} > 0$ exaggerates subtle differences.
- **The Fix**: Bar charts MUST start at 0. If you wish to show fine differences without starting at 0, use a dot plot or box plot where position—not length—encodes value.

#### 4. The 'Dynamite Plunger' Plot
- **The Flaw**: A bar indicating the mean with an upper error bar indicating standard deviation ($T$-shaped bar resembling a TNT detonator plunger). It completely obscures the underlying distribution! A bimodal distribution, a skewed distribution with extreme outliers, and a uniform distribution can all produce identical dynamite plots.
- **The Fix**: Box plots, violin plots, or strip plots with jittered individual data points.

### 4.3.2 Worked Example

In [ ]:
# Visual Demonstration: The Dynamite Plunger Plot vs Truth
# Let's create two synthetic experimental conditions with IDENTICAL mean and SD
# but completely different underlying biology!
np.random.seed(42)

# Condition A: Normal distribution around mean = 20
cond_a = np.random.normal(loc=20.0, scale=3.0, size=50)

# Condition B: Bimodal distribution (two distinct cell subpopulations) with SAME mean ~20
subpop1 = np.random.normal(loc=14.0, scale=1.0, size=25)
subpop2 = np.random.normal(loc=26.0, scale=1.0, size=25)
cond_b = np.concatenate([subpop1, subpop2])

print(f"Condition A: Mean={np.mean(cond_a):.2f}, SD={np.std(cond_a):.2f}")
print(f"Condition B: Mean={np.mean(cond_b):.2f}, SD={np.std(cond_b):.2f}")

# Plotting comparison
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), dpi=150)

# BAD: Dynamite Plunger Plot
axes[0].bar(["Group A", "Group B"], [np.mean(cond_a), np.mean(cond_b)], 
            yerr=[np.std(cond_a), np.std(cond_b)], capsize=8, color=['#b0c4de', '#b0c4de'], edgecolor='black')
axes[0].set_title("❌ Bad: Dynamite Plunger Plot\n(Hides Bimodal Subpopulations)", fontsize=11, fontweight="bold", color="darkred")
axes[0].set_ylabel("Marker Expression (AU)")
axes[0].set_ylim(0, 32)

# GOOD: Box Plot with Jittered Data Points
sns.boxplot(data=[cond_a, cond_b], ax=axes[1], palette=["#7293CB", "#E1974C"], boxprops=dict(alpha=0.6), width=0.4)
sns.stripplot(data=[cond_a, cond_b], ax=axes[1], color="black", size=6, jitter=0.2, alpha=0.7)
axes[1].set_xticklabels(["Group A", "Group B"])
axes[1].set_title("✔ Good: Box Plot + Individual Points\n(Reveals True Bimodality!)", fontsize=11, fontweight="bold", color="darkgreen")
axes[1].set_ylabel("Marker Expression (AU)")
axes[1].set_ylim(0, 32)

plt.tight_layout()
plt.show()

### 4.3.3 ✏️ Try It: Colormap Safety Auditor

Write a function `audit_colormap_safety(cmap_name: str) -> dict`:
- Return a dictionary with:
  - `"safe"`: `True` if `cmap_name` is in `["viridis", "plasma", "inferno", "magma", "cividis", "cubehelix"]`, else `False`.
  - `"reason"`: `"Perceptually uniform & colorblind-friendly"` if safe, else `"Rainbow/non-uniform colormap introduces perceptual artifacts and fails colorblind readers"`.
  - `"recommended_replacement"`: `"cividis"` if not safe, else `cmap_name`.

In [ ]:
def audit_colormap_safety(cmap_name: str) -> dict:
    safe_maps = {"viridis", "plasma", "inferno", "magma", "cividis", "cubehelix"}
    name = cmap_name.strip().lower()
    is_safe = name in safe_maps
    return {
        "safe": is_safe,
        "reason": "Perceptually uniform & colorblind-friendly" if is_safe else "Rainbow/non-uniform colormap introduces perceptual artifacts and fails colorblind readers",
        "recommended_replacement": name if is_safe else "cividis"
    }

#### 💡 Explanation & Design Notes

The Python `matplotlib` library switched its default colormap from `jet` to `viridis` in version 2.0 specifically because perceptual science demonstrated that `jet` introduces false edges and is uninterpretable for 8% of male readers.

In [ ]:
# ✅ Test — run me. No output means you passed.
res_jet = audit_colormap_safety("jet")
assert not res_jet["safe"]
assert res_jet["recommended_replacement"] == "cividis"

res_vir = audit_colormap_safety("viridis")
assert res_vir["safe"]
assert res_vir["recommended_replacement"] == "viridis"
print("Section 4A.3 passed ✔")

### 4.3.4 ⚠️ Common Mistakes

- **Relying on red-green contrasts for differential expression**: Standard red/green heatmaps are invisible to readers with deuteranopia or protanopia. Use blue/orange or magenta/green or viridis instead.
- **Creating 3D bar plots in Excel or software defaults**: Software that offers 3D rendering adds unnecessary visual clutter (chartjunk) that hinders quantitative extraction.
- **Overplotting raw scatter points without transparency**: When plotting 50,000 single cells or genomic loci, opaque points merge into a solid blob; use `alpha=0.2` or hexbin / 2D density plots.

### Practice Exercise 4A.1 (★) — Biological Variable Classification & Chart Recommendation

**Problem**: Given a dataset description dictionary, write a function `assign_plot_recommendation(meta: dict) -> str` that inspects the dataset characteristics and returns:
- `"Bar plot"` if `meta["goal"] == "compare_counts"` and `meta["categories"] <= 30`
- `"Histogram"` if `meta["goal"] == "distribution"` and `meta["var_type"] == "continuous"`
- `"Volcano plot"` if `meta["goal"] == "differential_expression"`
- `"Clustered Heatmap"` if `meta["goal"] == "expression_matrix"`
- `"Box plot"` if `meta["goal"] == "compare_distributions"`
- `"Invalid specification"` otherwise.

- **Input**: Metadata dictionary with keys 'goal', 'var_type', 'categories'
- **Expected Output**: String recommendation of plot type

In [ ]:
def assign_plot_recommendation(meta: dict) -> str:
    goal = meta.get("goal")
    var_type = meta.get("var_type")
    cats = meta.get("categories", 0)
    
    if goal == "compare_counts" and cats <= 30:
        return "Bar plot"
    elif goal == "distribution" and var_type == "continuous":
        return "Histogram"
    elif goal == "differential_expression":
        return "Volcano plot"
    elif goal == "expression_matrix":
        return "Clustered Heatmap"
    elif goal == "compare_distributions":
        return "Box plot"
    return "Invalid specification"

#### 💡 Solution Walkthrough (4A.1)

This exercise models a decision tree that enforces biological plot suitability. In complex bioinformatics pipelines, this logic can automatically route analytical outputs to their standard graphical representations.

In [ ]:
# ✅ Test — run me. No output means you passed.
assert assign_plot_recommendation({"goal": "compare_counts", "categories": 24}) == "Bar plot"
assert assign_plot_recommendation({"goal": "distribution", "var_type": "continuous"}) == "Histogram"
assert assign_plot_recommendation({"goal": "differential_expression"}) == "Volcano plot"
assert assign_plot_recommendation({"goal": "expression_matrix"}) == "Clustered Heatmap"
assert assign_plot_recommendation({"goal": "compare_distributions"}) == "Box plot"
assert assign_plot_recommendation({"goal": "unknown_goal"}) == "Invalid specification"
print("Ex 4A.1 passed ✔")

### Practice Exercise 4A.2 (★★) — Plot Health Auditor for Scientific Manuscripts

**Problem**: Write a function `audit_figure_spec(spec: dict) -> list[str]` that evaluates a dictionary specifying figure parameters and returns a list of warning strings:
- If `spec["type"] == "bar"` and `spec.get("y_min", 0) > 0`: add `"WARNING: Truncated y-axis on bar chart distorts magnitude ratios."`
- If `spec.get("is_3d", False)`: add `"WARNING: 3D perspective causes optical distortion and occlusion."`
- If `spec.get("colormap") in ["jet", "rainbow", "hsv"]`: add `"WARNING: Rainbow colormap is non-uniform and colorblind-hostile."`
- If `spec["type"] == "pie"` and `spec.get("num_slices", 0) > 5`: add `"WARNING: Pie chart has excessive slices (>5); replace with bar chart."`
- If no warnings are detected, return `["PASS: Specification follows data visualization best practices."]`.


- **Input**: Dictionary specifying 'type', 'y_min', 'is_3d', 'colormap', 'num_slices'
- **Expected Output**: List of warning strings or PASS confirmation

In [ ]:
def audit_figure_spec(spec: dict) -> list[str]:
    warnings = []
    if spec.get("type") == "bar" and spec.get("y_min", 0) > 0:
        warnings.append("WARNING: Truncated y-axis on bar chart distorts magnitude ratios.")
    if spec.get("is_3d", False):
        warnings.append("WARNING: 3D perspective causes optical distortion and occlusion.")
    if spec.get("colormap", "").lower() in ["jet", "rainbow", "hsv"]:
        warnings.append("WARNING: Rainbow colormap is non-uniform and colorblind-hostile.")
    if spec.get("type") == "pie" and spec.get("num_slices", 0) > 5:
        warnings.append("WARNING: Pie chart has excessive slices (>5); replace with bar chart.")
    
    if not warnings:
        return ["PASS: Specification follows data visualization best practices."]
    return warnings

#### 💡 Solution Walkthrough (4A.2)

Automatic figure auditing protects computational biology researchers from peer review rejections. Modern top journals like Nature Methods routinely enforce guidelines against truncated bar charts and rainbow colormaps.

In [ ]:
# ✅ Test — run me. No output means you passed.
bad_spec = {"type": "bar", "y_min": 50, "is_3d": True, "colormap": "jet"}
w = audit_figure_spec(bad_spec)
assert len(w) == 3
assert any("Truncated" in item for item in w)
assert any("3D" in item for item in w)
assert any("Rainbow" in item for item in w)

good_spec = {"type": "bar", "y_min": 0, "is_3d": False, "colormap": "viridis", "num_slices": 4}
w_good = audit_figure_spec(good_spec)
assert len(w_good) == 1 and "PASS" in w_good[0]
print("Ex 4A.2 passed ✔")

### Practice Exercise 4A.3 (★★★) — Generating a Publication-Quality Comparison Figure

**Problem**: Write a function `generate_bad_vs_good_plot(out_path: Path) -> Path` that creates a 2-panel figure:
- Panel A (Left): A dynamite plunger plot (bar chart of mean + SD) of two synthetic populations `A` and `B`.
- Panel B (Right): A box plot with overlaid jittered points (`sns.stripplot`) showing the true distribution of `A` (unimodal, normal) and `B` (bimodal).
- Saves the figure to `out_path` with `dpi=300` and `bbox_inches='tight'`.
- Returns the `Path` to the saved file.


- **Input**: Path object destination
- **Expected Output**: Saved figure file path with DPI=300

In [ ]:
def generate_bad_vs_good_plot(out_path: Path) -> Path:
    np.random.seed(42)
    pop_a = np.random.normal(loc=25.0, scale=4.0, size=60)
    sub1 = np.random.normal(loc=17.0, scale=1.5, size=30)
    sub2 = np.random.normal(loc=33.0, scale=1.5, size=30)
    pop_b = np.concatenate([sub1, sub2])
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), dpi=300)
    
    # Left: Bad dynamite plot
    means = [np.mean(pop_a), np.mean(pop_b)]
    sds = [np.std(pop_a), np.std(pop_b)]
    axes[0].bar(["Group A", "Group B"], means, yerr=sds, capsize=6, color="#C0C0C0", edgecolor="black")
    axes[0].set_title("A. Misleading Dynamite Plot (Mean ± SD)", fontsize=10, fontweight="bold")
    axes[0].set_ylabel("Abundance (AU)")
    axes[0].set_ylim(0, 42)
    
    # Right: Good box plot + jittered strip plot
    sns.boxplot(data=[pop_a, pop_b], ax=axes[1], palette=["#4C72B0", "#55A868"], boxprops=dict(alpha=0.5), width=0.4)
    sns.stripplot(data=[pop_a, pop_b], ax=axes[1], color="black", size=5, alpha=0.7, jitter=0.2)
    axes[1].set_xticklabels(["Group A", "Group B"])
    axes[1].set_title("B. Informative Boxplot + Jittered Raw Points", fontsize=10, fontweight="bold")
    axes[1].set_ylabel("Abundance (AU)")
    axes[1].set_ylim(0, 42)
    
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return out_path

#### 💡 Solution Walkthrough (4A.3)

This function programmatically renders both plots using Matplotlib and Seaborn, applies appropriate styling and labels, exports to high-resolution PNG (300 DPI), and verifies file generation on disk.

In [ ]:
# ✅ Test — run me. No output means you passed.
target_file = OUT_DIR / "bad_vs_good_plot.png"
saved = generate_bad_vs_good_plot(target_file)
assert saved.exists(), "Target file was not saved!"
assert saved.stat().st_size > 10000, "Figure file is too small or empty!"
print(f"Ex 4A.3 passed ✔ Saved to {saved}")

## 6. Summary & Cheat Sheet

In this notebook, we established the theoretical foundation of biological data visualization. Selecting the right chart type is determined by your biological hypothesis and data structure (continuous vs categorical, 1D vs 2D vs matrix). Avoiding deceptive plotting pitfalls (such as dynamite plungers and rainbow colormaps) guarantees scientific clarity and accessibility.

### Quick Reference Table

| Concept / Operation | Syntax / Function | Biological Application | Key Parameter / Note |
| --- | --- | --- | --- |
| Bar Plot | plt.bar / sns.barplot | Discrete category comparisons (e.g. chromosome counts) | Baseline MUST start at 0 |
| Histogram / KDE | plt.hist / sns.histplot | Distribution shape of continuous variable (e.g. GC%) | Set bin size thoughtfully |
| Box Plot | plt.boxplot / sns.boxplot | Median & spread across conditions with outliers | Always show raw points if N < 30 |
| Violin Plot | plt.violinplot / sns.violinplot | Multimodal / single-cell distributions | Combines KDE with IQR quartiles |
| Volcano Plot | plt.scatter (log2FC vs -log10 p) | Differential expression significance vs magnitude | Annotate top regulated genes |
| Clustered Heatmap | sns.clustermap | Multi-sample expression matrices | Row/column z-score standardization |
| Perceptually Uniform Maps | viridis, plasma, cividis | All continuous heatmaps & density plots | Safe for colorblind readers & greyscale |

## 7. Further Reading & Resources

- [Nature Methods: Points of Significance (Visualizing Data)](https://www.nature.com/articles/nmeth.2613): Seminal guide by Krzywinski & Altman on choosing proper data representations.
- [Nature Methods: Points of Significance (Color Blindness)](https://www.nature.com/articles/nmeth.1618): Detailed exploration of color perception and accessible palettes.
- [Ten Simple Rules for Better Figures](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1003833): Rougier et al., PLoS Computational Biology (2014).
- [Fundamentals of Data Visualization](https://clauswilke.com/): Claus O. Wilke's comprehensive open-access textbook on scientific plotting.